## FastText Fine-Tuned

Fine-tuning model pre-trained FastText (`cc.id.300.bin`) menggunakan data domain spesifik.

**Tujuan:** Model pre-trained dilatih dari Wikipedia/Common Crawl (general), sehingga vektor kata seperti *"jaringan"* mungkin lebih dekat ke konteks biologi. Setelah fine-tuning dengan data kamu, vektor bergeser ke konteks yang relevan dengan datamu.

**Alur Fine-Tuning:**
1. Load pre-trained model (`cc.id.300.bin`) sebagai titik awal  
2. Siapkan corpus dari data kamu (soal + kunci + jawaban siswa)  
3. Lanjutkan training (`fine-tune`) pada corpus domain  
4. Simpan model fine-tuned  
5. Gunakan model fine-tuned untuk embedding

### Setup & Install Dependencies

In [ ]:
# Install jika belum ada
# %pip install gensim numpy pandas tqdm

In [1]:
import os
import ast
import numpy as np
import pandas as pd
from tqdm import tqdm
from gensim.models.fasttext import load_facebook_model, FastText

print("Semua library berhasil di-import!")
print(f"Gensim version: {__import__('gensim').__version__}")

Semua library berhasil di-import!
Gensim version: 4.3.3


### Load Pre-trained FastText Model

Model `cc.id.300.bin` dimuat menggunakan **gensim** (`load_facebook_model`).  
Gensim mendukung fine-tuning (lanjut training) langsung dari model Facebook FastText binary.

In [2]:
model_path = r'Pre-trained FastText Model\cc.id.300.bin'

print("Memuat pre-trained FastText model (ini butuh waktu beberapa menit)...")
ft_model = load_facebook_model(model_path)
print("Model berhasil di-load!")
print(f"Dimensi vektor : {ft_model.wv.vector_size}")
print(f"Jumlah vocab   : {len(ft_model.wv)}")

Memuat pre-trained FastText model (ini butuh waktu beberapa menit)...
Model berhasil di-load!
Dimensi vektor : 300
Jumlah vocab   : 2000000


### Persiapan Corpus dari Data Domain

Semua teks dari dataset (soal, kunci jawaban, jawaban siswa) digabung menjadi corpus untuk fine-tuning.  
Corpus berbentuk **list of list of words** — format yang dibutuhkan FastText.

In [3]:
df = pd.read_csv('preprocessed.csv')
print(f"Dataset dimuat: {len(df)} baris")
print(f"Kolom: {list(df.columns)}\n")

# Parse kolom list (disimpan sebagai string di CSV)
for col in ['questions_clean', 'answerKeys_clean', 'answer_clean']:
    df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

def text_to_words(text_list):
    """Ubah list of sentences menjadi list of words."""
    if not text_list:
        return []
    combined = ' '.join([str(s) for s in text_list if s and not pd.isna(s)])
    return combined.split()

# Bangun corpus: setiap kalimat / dokumen = 1 list of words
corpus = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Membangun corpus"):
    q_words   = text_to_words(row['questions_clean'])
    ak_words  = text_to_words(row['answerKeys_clean'])
    ans_words = text_to_words(row['answer_clean'])

    if q_words:
        corpus.append(q_words)
    if ak_words:
        corpus.append(ak_words)
    if ans_words:
        corpus.append(ans_words)

# Hapus duplikat persis (opsional, menghemat waktu training)
corpus = [list(x) for x in set(tuple(s) for s in corpus)]

print(f"\nTotal kalimat / dokumen dalam corpus : {len(corpus)}")
total_words = sum(len(s) for s in corpus)
print(f"Total kata dalam corpus              : {total_words}")
print(f"\nContoh 3 kalimat pertama:")
for i, sentence in enumerate(corpus[:3]):
    print(f"  [{i}] {sentence}")

Dataset dimuat: 817 baris
Kolom: ['IDJwb', 'IDPSJ', 'questions_clean', 'answerKeys_clean', 'answer_clean', 'raw_grade', 'max_grade', 'grade', 'labela', 'label']



Membangun corpus: 100%|██████████| 817/817 [00:00<00:00, 11751.05it/s]


Total kalimat / dokumen dalam corpus : 844
Total kata dalam corpus              : 31611

Contoh 3 kalimat pertama:
  [0] ['critical', 'section', 'adalah', 'dengan', 'mendesain', 'sebuah', 'protokol', 'di', 'mana', 'proses', 'proses', 'dapat', 'menggunakannya', 'secara', 'bersama', 'sama', 'setiap', 'proses', 'harus', 'â€˜meminta', 'izinâ€™', 'untuk', 'memasuki', 'critical', 'section', 'nya', 'bagian', 'dari', 'kode', 'yang', 'mengimplementasikan', 'izin', 'ini', 'disebut', 'entry', 'section', 'akhir', 'dari', 'critical', 'section', 'itu', 'disebut', 'exit', 'section', 'bagian', 'kode', 'selanjutnya', 'disebut', 'remainder', 'section']
  [1] ['sumpah', 'pemuda', 'kami', 'putra', 'dan', 'putri', 'indonesia', 'bersumpah', 'bernegara', 'yang', 'satu', 'tanah', 'air', 'indonesia', 'kami', 'putra', 'dan', 'putri', 'indonesia', 'bersumpah', 'bertumpah', 'darah', 'yang', 'satu', 'tanah', 'air', 'indonesia', 'kami', 'putra', 'dan', 'putri', 'indonesia', 'bersumpah', 'berbahasa', 'yang', 'satu'

### Fine-Tuning

Proses fine-tuning menggunakan dua langkah gensim:  
1. `build_vocab(corpus, update=True)` → tambahkan kata-kata baru dari domain ke vocab model  
2. `train(corpus, ...)` → lanjutkan training, perbarui bobot vektor berdasarkan corpus domain

Parameter penting:
| Parameter  | Nilai | Keterangan |
|------------|-------|------------|
| `epochs`   | 10    | Iterasi training (naikkan ke 20-30 jika hasil kurang baik) |
| `min_count`| 1     | Kata minimal muncul 1x agar masuk vocab baru |
| `workers`  | 4     | Jumlah thread parallel |

In [ ]:
import time

# ===== KONFIGURASI FINE-TUNING =====
EPOCHS    = 10    # Jumlah iterasi training
MIN_COUNT = 1     # Kata baru minimal muncul N kali agar masuk vocab
WORKERS   = 4     # Thread parallel

print("="*60)
print("FINE-TUNING FastText")
print("="*60)
print(f"Corpus size : {len(corpus)} kalimat")
print(f"Epochs      : {EPOCHS}")
print(f"Min count   : {MIN_COUNT}")
print()

# Step 1: Update vocab dengan kata-kata baru dari corpus domain
print("Step 1/2 - Memperbarui vocabulary...")
ft_model.build_vocab(corpus, update=True)
print(f"Vocab setelah update: {len(ft_model.wv)} kata")
print()

# Step 2: Fine-tune (lanjut training)
ft_model.workers = WORKERS

print("Step 2/2 - Fine-tuning model...")
start = time.time()

ft_model.train(
    corpus,
    total_examples=len(corpus),
    epochs=EPOCHS,
    start_alpha=0.005,
    end_alpha=0.0001
)

elapsed = time.time() - start
print(f"\nFine-tuning selesai dalam {elapsed:.1f} detik ({elapsed/60:.1f} menit)")
print(f"Dimensi vektor : {ft_model.wv.vector_size}")
print(f"Total vocab    : {len(ft_model.wv)}")

FINE-TUNING FastText
Corpus size : 844 kalimat
Epochs      : 10
Min count   : 1

Step 1/2 - Memperbarui vocabulary...
Vocab setelah update: 2000010 kata

Step 2/2 - Fine-tuning model...

Fine-tuning selesai dalam 29.1 detik (0.5 menit)
Dimensi vektor : 300
Total vocab    : 2000010


### Simpan Model Fine-Tuned

In [6]:
save_path = 'fasttext_finetuned.model'

ft_model.save(save_path)
print(f"Model fine-tuned disimpan ke: {save_path}")
print()
# print("Cara load kembali:")
# print(f"  ft_model = FastText.load('{save_path}')")

Model fine-tuned disimpan ke: fasttext_finetuned.model



### Validasi: Kata-Kata Paling Mirip (Fine-Tuned Model)

Cek apakah model sudah mengenal konteks domain dengan melihat kata-kata terdekat untuk beberapa kata kunci dari datamu.

> **Catatan**: Untuk membandingkan dengan pre-trained asli, kamu perlu load ulang `cc.id.300.bin` ke variabel terpisah (butuh RAM ekstra ~4GB).

In [ ]:
# Ganti kata-kata ini dengan kata kunci penting dari domain datamu
# Contoh kata-kata yang relevan dengan data
kata_kunci = [
    'jaringan',    # → setelah fine-tune seharusnya lebih dekat ke konteks data
    'karakter',
    'pikir',
    'gotong',
    'kerja',
]

print("="*60)
print("TOP-5 KATA PALING MIRIP (Model Fine-Tuned)")  
print("="*60)

for kata in kata_kunci:
    try:
        similar = ft_model.wv.most_similar(kata, topn=5)
        print(f"\n'{kata}':")
        for sim_word, score in similar:
            print(f"  {sim_word:<25} {score:.4f}")
    except KeyError:
        print(f"\n'{kata}': tidak ada dalam vocab")

### Fungsi Embedding untuk BiLSTM (Sequence-based)

Sama persis seperti di `fasttext.ipynb`, tapi menggunakan model fine-tuned.  
Kata OOV (Out-of-Vocabulary) tetap mendapat vektor dari subword/karakter — salah satu keunggulan FastText.

In [ ]:
# ===== KONFIGURASI SEQ_LEN =====
# Sesuaikan dengan hasil analisis distribusi token di fasttext.ipynb
SEQ_LEN_QUESTIONS  = 22
SEQ_LEN_ANSWERKEYS = 38
SEQ_LEN_ANSWERS    = 85

EMBEDDING_DIM = ft_model.wv.vector_size  # 300

def get_word_vector(word):
    """
    Ambil vektor kata dari model fine-tuned.
    Gensim FastText otomatis menggunakan subword untuk OOV.
    """
    return ft_model.wv[word]  # OOV → subword vector secara otomatis


def get_sequence_embedding(sentences, seq_len):
    """
    Ubah dokumen (list of sentences) menjadi sequence embedding untuk BiLSTM.
    
    Alur:
    1. Tokenizing  : Teks → list of words
    2. Truncation  : Potong jika lebih dari seq_len
    3. Embedding   : Setiap word → vektor fine-tuned (300 dim)
    4. Padding     : Zero-pad jika kurang dari seq_len
    
    Returns:
        numpy array (seq_len, embedding_dim)
    """
    if not sentences or len(sentences) == 0:
        return np.zeros((seq_len, EMBEDDING_DIM))

    if isinstance(sentences, list):
        all_text = ' '.join([str(s) for s in sentences if s and not pd.isna(s)])
    else:
        all_text = str(sentences) if not pd.isna(sentences) else ''

    words = all_text.split()

    # Truncation
    if len(words) > seq_len:
        words = words[:seq_len]

    # Embedding
    sequence = np.zeros((seq_len, EMBEDDING_DIM))
    for i, word in enumerate(words):
        sequence[i] = get_word_vector(word)

    return sequence


# Wrapper per kolom
def get_questions_embedding(sentences):
    return get_sequence_embedding(sentences, SEQ_LEN_QUESTIONS)

def get_answerkeys_embedding(sentences):
    return get_sequence_embedding(sentences, SEQ_LEN_ANSWERKEYS)

def get_answers_embedding(sentences):
    return get_sequence_embedding(sentences, SEQ_LEN_ANSWERS)

print(f"Fungsi embedding siap. Embedding dim = {EMBEDDING_DIM}")
print(f"SEQ_LEN: questions={SEQ_LEN_QUESTIONS}, answerKeys={SEQ_LEN_ANSWERKEYS}, answers={SEQ_LEN_ANSWERS}")

### Proses Embedding untuk Semua Data

In [ ]:
from tqdm import tqdm
tqdm.pandas()

print("="*60)
print("MEMPROSES EMBEDDING (Fine-Tuned Model)")
print("="*60)
print(f"\nTotal data: {len(df)} baris\n")

# ----- 1. Questions & AnswerKeys (per unique IDPSJ) -----
print("1/2 - Memproses Questions & AnswerKeys (unique IDPSJ)...")
unique_idpsj = df.drop_duplicates(subset='IDPSJ')[['IDPSJ', 'questions_clean', 'answerKeys_clean']].copy()
print(f"  → {len(unique_idpsj)} unique IDPSJ\n")

unique_idpsj['questions_embedding']  = unique_idpsj['questions_clean'].progress_apply(get_questions_embedding)
unique_idpsj['answerKeys_embedding'] = unique_idpsj['answerKeys_clean'].progress_apply(get_answerkeys_embedding)

# Merge kembali ke df
df = df.merge(
    unique_idpsj[['IDPSJ', 'questions_embedding', 'answerKeys_embedding']],
    on='IDPSJ', how='left'
)
print("  ✓ Questions & AnswerKeys selesai!\n")

# ----- 2. Student Answers (semua baris) -----
print("2/2 - Memproses Student Answers...")
df['answer_embedding'] = df['answer_clean'].progress_apply(get_answers_embedding)
print("  ✓ Student Answers selesai!\n")

# ----- Verifikasi -----
print("="*60)
print("SEMUA EMBEDDING SELESAI")
print("="*60)
print(f"Jumlah baris           : {len(df)}")
print(f"questions_embedding    : {df['questions_embedding'].iloc[0].shape}")
print(f"answerKeys_embedding   : {df['answerKeys_embedding'].iloc[0].shape}")
print(f"answer_embedding       : {df['answer_embedding'].iloc[0].shape}")

### Simpan & Load Hasil Embedding

In [ ]:
# Pilih kolom yang relevan dan simpan ke pickle
df_out = df[['IDJwb', 'IDPSJ',
             'questions_embedding', 'answerKeys_embedding', 'answer_embedding',
             'raw_grade', 'max_grade', 'grade', 'labela', 'label']]

output_pkl = 'df_embeddings_finetuned.pkl'
df_out.to_pickle(output_pkl)
print(f"Embedding disimpan ke: {output_pkl}")

In [ ]:
# Load model fine-tuned (jika mau dipakai ulang tanpa re-training)
# from gensim.models.fasttext import FastText
# ft_model = FastText.load('fasttext_finetuned.model')

# Load embedding dari pickle
df_loaded = pd.read_pickle('df_embeddings_finetuned.pkl')
print(f"DataFrame dimuat: {len(df_loaded)} baris")
print(f"Kolom: {list(df_loaded.columns)}")
print(f"\nShape embedding:")
print(f"  questions_embedding  : {df_loaded['questions_embedding'].iloc[0].shape}")
print(f"  answerKeys_embedding : {df_loaded['answerKeys_embedding'].iloc[0].shape}")
print(f"  answer_embedding     : {df_loaded['answer_embedding'].iloc[0].shape}")

print(f"\nContoh 5 vektor pertama dari answer_embedding[0]:")
print(df_loaded['answer_embedding'].iloc[0][:5])